In [65]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


In [ ]:
# dataset paths
DATASET_ROOT = Path("~/Datasets/ADNI/ADNI/AD_NC").expanduser()
METADATA_PATH = Path("~/Datasets/ADNI/ADNI/meta_data_with_label.json").expanduser()

SEED = 42
CLASS_NAMES = ["AD", "NC"]
LABEL_MAP = {"NC": 0, "AD": 1}

In [67]:
with METADATA_PATH.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

# regex pattern for matching patient id from meta data
PATIENT_PATTERN = re.compile(r"ADNI_(\d{3}_S_\d{4})", re.IGNORECASE)

# read patient id from raw key in meta data
def patient_id_from_record(record):
    value = record.get('raw')
    match = PATIENT_PATTERN.search(value)
    return match.group(1).upper()

scan_to_patient = {
    str(scan_id): patient_id_from_record(record)
    for scan_id, record in metadata.items()
}

known_patients = {patient for patient in scan_to_patient.values() if patient is not None}
print(f"Metadata records: {len(metadata):,}")
print(f"Patients in metadata: {len(known_patients):,}")

Metadata records: 2,189
Patients in metadata: 942


In [68]:
# create a table containing all images from the dataset
def build_image_table(dataset_root, scan_to_patient):
    rows = []

    for original_split in ["train", "test"]:
        for class_name in CLASS_NAMES:
            folder = dataset_root / original_split / class_name
            for path in sorted(folder.rglob("*")):
                scan_id = str(path).split("/")[-1].split("_")[0]
                rows.append({
                    "image_path": str(path.resolve()),
                    "image_name": path.name,
                    "original_split": original_split,
                    "class_name": class_name,
                    "label": LABEL_MAP[class_name],
                    "scan_id": scan_id,
                    "patient_id": scan_to_patient.get(scan_id),
                })

    table = pd.DataFrame(rows)
    return table

dataset_df = build_image_table(DATASET_ROOT, scan_to_patient)

print(f"Total images: {len(dataset_df)}")
print(f"Total scans: {dataset_df['scan_id'].nunique()}")
print(f"Total patients: {dataset_df['patient_id'].nunique()}")
display(dataset_df.head())


Total images: 30520
Total scans: 1526
Total patients: 680


,image_path,image_name,original_split,class_name,label,scan_id,patient_id
0,/Users/chris/Datasets/ADNI/ADNI/AD_NC/train/AD...,218391_78.jpeg,train,AD,1,218391,037_S_4001
1,/Users/chris/Datasets/ADNI/ADNI/AD_NC/train/AD...,218391_79.jpeg,train,AD,1,218391,037_S_4001
2,/Users/chris/Datasets/ADNI/ADNI/AD_NC/train/AD...,218391_80.jpeg,train,AD,1,218391,037_S_4001
3,/Users/chris/Datasets/ADNI/ADNI/AD_NC/train/AD...,218391_81.jpeg,train,AD,1,218391,037_S_4001
4,/Users/chris/Datasets/ADNI/ADNI/AD_NC/train/AD...,218391_82.jpeg,train,AD,1,218391,037_S_4001


In [69]:
# create table where each row is one patient
patients = dataset_df[["patient_id", "class_name"]].drop_duplicates("patient_id")

# split patients into 70% training, and 30% for validation and testing
train, remaining = train_test_split(
    patients,
    test_size=0.3,
    stratify=patients["class_name"],
    random_state=42
)

# split remaining 30% of patients into 20% validation and 10% testing
val, test = train_test_split(
    remaining,
    test_size=1/3,
    stratify=remaining["class_name"],
    random_state=42
)

# assign splits to images based on patient ID
dataset_df["split"] = "train"
dataset_df.loc[dataset_df["patient_id"].isin(val["patient_id"]), "split"] = "validation"
dataset_df.loc[dataset_df["patient_id"].isin(test["patient_id"]), "split"] = "test"

# separate dataframes
train_df = dataset_df[dataset_df["split"] == "train"].reset_index(drop=True)
val_df = dataset_df[dataset_df["split"] == "validation"].reset_index(drop=True)
test_df = dataset_df[dataset_df["split"] == "test"].reset_index(drop=True)

In [70]:
total_images = len(train_df) + len(val_df) + len(test_df)

print(f"Training Images: {len(train_df)} - {len(train_df) / total_images:.2%} of dataset")
print(f"Validation Images: {len(val_df)} - {len(val_df) / total_images:.2%} of dataset")
print(f"Test Images: {len(test_df)} - {len(test_df) / total_images:.2%} of dataset")

print(f"Total Images: {total_images}")


Training Images: 21140 - 69.27% of dataset
Validation Images: 6420 - 21.04% of dataset
Test Images: 2960 - 9.70% of dataset
Total Images: 30520


In [71]:
# check class proportions in each split
for split in ["train", "validation", "test"]:
    split_data = dataset_df[dataset_df["split"] == split]
    print(f"\n{split}")
    print(split_data["class_name"].value_counts())
    print(f"Total: {len(split_data)}")


train
class_name
NC    10660
AD    10480
Name: count, dtype: int64
Total: 21140

validation
class_name
NC    3440
AD    2980
Name: count, dtype: int64
Total: 6420

test
class_name
NC    1560
AD    1400
Name: count, dtype: int64
Total: 2960


Spltting with stratification by patient ID has lead to effective splitting.

Each split has nearly equal class proportions and the splits are almost exactly 70%, 20%, and 10% respectively.